# Лабораторная работа 2
## Синонимическое обезличивание персональных данных

**Цель:** реализовать программу, которая получает текст и координаты ПДн из ЛР1 и возвращает текст со случайными заменами того же типа.

Работа выполнена по требованиям файла «Лабораторная работа 2.docx». Используются `pymorphy3`, заранее подготовленные списки имён и фамилий и `SystemRandom`, как рекомендовано в методичке. Распознавание ПДн относится к ЛР1; здесь обрабатываются переданные интервалы.

| Атрибут | Что сохраняется |
|---|---|
| Фамилия | Род, число, падеж |
| Имя | Род, число, падеж |
| E-mail | Домен после `@` |
| ИНН | Первые четыре цифры; контрольные разряды остаются валидными |
| Телефон | Первые четыре цифры и все символы, кроме цифр |
| Банковская карта | Первые шесть и последние четыре цифры; проверка Луна |

**Запуск в Google Colab:** загрузить этот файл через «Файл → Открыть блокнот → Загрузить» и выбрать «Среда выполнения → Выполнить всё». Достаточно CPU. Первая ячейка устанавливает единственную внешнюю библиотеку; интернет нужен для её установки. Все остальные данные и код находятся в ноутбуке.

In [1]:
%pip -q install pymorphy3==2.0.6

### 1. Подготовка и проверка контрольных разрядов

`digits` убирает разделители только на время вычислений, а `restore_format` возвращает их на прежние позиции. В ИНН из 10 цифр один контрольный разряд, в ИНН из 12 цифр — два. Для каждого разряда считается взвешенная сумма, затем остатки `% 11 % 10`.

В алгоритме Луна цифры на чётных позициях справа удваиваются; из результатов больше 9 вычитается 9. Номер проходит проверку, если итоговая сумма делится на 10.

In [2]:
import re
import string
from random import SystemRandom
import pymorphy3

rng = SystemRandom()  # Без фиксированного seed.
morph = pymorphy3.MorphAnalyzer()


def digits(value):
    return ''.join(c for c in value if c in string.digits)


def restore_format(template, new_digits):
    """Подставить цифры, оставив все разделители на своих местах."""
    if len(digits(template)) != len(new_digits):
        raise ValueError('Количество цифр не совпадает с исходным форматом')
    iterator = iter(new_digits)
    return ''.join(next(iterator) if c in string.digits else c for c in template)


def checksum(number, weights):
    return str(sum(int(d) * w for d, w in zip(number, weights)) % 11 % 10)


def complete_inn(body):
    """Добавить контрольные разряды к основе из 9 или 10 цифр."""
    if len(body) == 9:
        return body + checksum(body, [2, 4, 10, 3, 5, 9, 4, 6, 8])
    if len(body) == 10:
        number = body + checksum(body, [7, 2, 4, 10, 3, 5, 9, 4, 6, 8])
        return number + checksum(number, [3, 7, 2, 4, 10, 3, 5, 9, 4, 6, 8])
    raise ValueError('Основа ИНН должна содержать 9 или 10 цифр')


def valid_inn(number):
    if not re.fullmatch(r'[0-9]{10}|[0-9]{12}', number):
        return False
    body = number[:-1] if len(number) == 10 else number[:-2]
    return len(set(number)) > 1 and complete_inn(body) == number


def luhn_sum(number):
    total = 0
    for i, digit in enumerate(reversed(number)):
        value = int(digit)
        if i % 2 == 1:
            value *= 2
            if value > 9:
                value -= 9
        total += value
    return total


def valid_card(number):
    return bool(re.fullmatch(r'[0-9]{13,19}', number)) and luhn_sum(number) % 10 == 0

### 2. Обезличивание структурированных атрибутов

В ИНН случайно меняется часть после первых четырёх цифр и пересчитываются контрольные разряды. В телефоне меняются цифры после первых четырёх, оформление сохраняется. В e-mail меняются буквы и цифры логина, домен сохраняется посимвольно.

У карты **последние четыре цифры, включая контрольную, не меняются**. Случайно формируется середина, затем одна из её цифр подбирается перебором от 0 до 9 до выполнения проверки Луна. Это соответствует требованию методички и примеру из лекции 3.

Новая сущность всегда отличается от исходной. Используется `SystemRandom` без фиксированного seed. Независимые случайные вызовы могут совпасть между собой, особенно для короткого списка имён; случайность не означает уникальность всех результатов.

In [3]:
def numeric_value(value, label):
    pattern = r'\+?[0-9\s()./–—-]+' if label == 'PHONE' else r'[0-9\s-]+'
    if not isinstance(value, str) or not re.fullmatch(pattern, value):
        raise ValueError(f'{label}: недопустимый формат')
    return digits(value)


def anon_inn(value):
    number = numeric_value(value, 'INN')
    if not valid_inn(number):
        raise ValueError('INN: нужны 10 или 12 цифр с верными контрольными разрядами')
    body_length = 9 if len(number) == 10 else 10
    while True:
        body = number[:4] + ''.join(rng.choice(string.digits) for _ in range(body_length - 4))
        new = complete_inn(body)
        if new != number:
            return restore_format(value, new)


def anon_phone(value):
    number = numeric_value(value, 'PHONE')
    if not 10 <= len(number) <= 15:
        raise ValueError('PHONE: ожидается от 10 до 15 цифр')
    while True:
        new = number[:4] + ''.join(rng.choice(string.digits) for _ in number[4:])
        if new != number:
            return restore_format(value, new)


def anon_card(value):
    number = numeric_value(value, 'CARD')
    if not valid_card(number):
        raise ValueError('CARD: нужны 13–19 цифр и корректная проверка Луна')
    # Меняем только середину [6:-4]. Последняя цифра карты остаётся прежней.
    # Одну цифру середины перебираем: ровно одно из 10 значений даёт нужную сумму.
    while True:
        middle = ''.join(rng.choice(string.digits) for _ in range(len(number) - 11))
        for correction in string.digits:
            new = number[:6] + middle + correction + number[-4:]
            if new != number and luhn_sum(new) % 10 == 0:
                return restore_format(value, new)


def anon_email(value):
    if not isinstance(value, str) or value.count('@') != 1:
        raise ValueError('EMAIL: требуется один символ @')
    local, domain = value.split('@')
    domain_labels = domain.split('.')
    if (not re.fullmatch(r'[A-Za-z0-9._%+\-]{1,64}', local)
            or local.startswith('.') or local.endswith('.') or '..' in local
            or not any(c.isalnum() for c in local)
            or len(domain_labels) < 2
            or any(not re.fullmatch(r'[^\W_](?:[^\W_]|-){0,61}[^\W_]|[^\W_]', x)
                   for x in domain_labels)):
        raise ValueError('EMAIL: ожидается обычный адрес вида user@example.com')
    result = []
    for c in local:
        alphabet = (string.ascii_lowercase if c.islower() else
                    string.ascii_uppercase if c.isupper() else
                    string.digits if c.isdigit() else '')
        result.append(rng.choice(alphabet.replace(c, '')) if alphabet else c)
    return ''.join(result) + '@' + domain

### 3. Обезличивание имён и фамилий

Из разборов `pymorphy3` выбирается разбор с признаком `Name` или `Surn`. Затем выбирается другое имя или фамилия, которую можно поставить в тот же род, число и падеж. Регистр и дефисы сохраняются дополнительно.

Некоторые формы неоднозначны: например, «Анне» может быть дательным или предложным падежом. Для нескольких однозначных предлогов используется подсказка из текста, иначе выбирается наиболее вероятный словарный разбор. Анализатор не выполняет полный синтаксический анализ предложения. Если подходящего разбора или замены нет, выдаётся `ValueError`.

In [4]:
NAMES = {
    'masc': ['Иван', 'Пётр', 'Андрей', 'Дмитрий', 'Сергей', 'Алексей', 'Николай', 'Михаил'],
    'femn': ['Анна', 'Мария', 'Ольга', 'Елена', 'Наталья', 'Ирина', 'Татьяна', 'Светлана'],
    'ms-f': ['Саша', 'Женя', 'Валя'],
}
SURNAMES = ['Иванов', 'Петров', 'Сидоров', 'Смирнов', 'Соколов', 'Волков',
            'Орлов', 'Козлов', 'Морозов', 'Новиков', 'Фёдоров', 'Попов']


def apply_case(template, word):
    if template.isupper():
        return word.upper()
    if template.islower():
        return word.lower()
    if template.istitle():
        return word.title()
    raise ValueError('Имя/фамилия: смешанный регистр не поддерживается')


def parse_person(word, kind, case_hint=None):
    variants = [p for p in morph.parse(word) if kind in p.tag
                and p.is_known and 'Abbr' not in p.tag]
    if case_hint:
        suitable = [p for p in variants if p.tag.case == case_hint]
        if suitable:
            variants = suitable
    if not variants:
        raise ValueError('Имя/фамилия не найдены в морфологическом словаре')
    return max(variants, key=lambda p: p.score)


def anon_person(value, kind, case_hint=None):
    if not isinstance(value, str) or not re.fullmatch(r'[А-Яа-яЁё]+(?:-[А-Яа-яЁё]+)*', value):
        raise ValueError('Имя/фамилия: ожидается русское слово или части через дефис')
    result = []
    for word in value.split('-'):
        source = parse_person(word, kind, case_hint)
        gender = 'ms-f' if 'ms-f' in source.tag else source.tag.gender
        number, case = source.tag.number, source.tag.case
        required = {x for x in (gender, number, case) if x}
        # Род ms-f у фамилий во множественном числе не задаётся при склонении.
        if kind == 'Surn' and number == 'plur':
            required.discard('ms-f')
        bases = NAMES.get(gender, []) if kind == 'Name' else SURNAMES
        options = set()
        for base in bases:
            for parsed in morph.parse(base):
                if kind not in parsed.tag or parsed.tag.case != 'nomn':
                    continue
                inflected = parsed.inflect(required)
                if inflected is None or not required.issubset(inflected.tag.grammemes):
                    continue
                if (inflected.normal_form.replace('ё', 'е') == source.normal_form.replace('ё', 'е')
                        or inflected.word.replace('ё', 'е') == word.lower().replace('ё', 'е')):
                    continue
                options.add(apply_case(word, inflected.word))
        if not options:
            raise ValueError('Нет замены с нужными родом, числом и падежом')
        result.append(rng.choice(sorted(options)))
    return '-'.join(result)

### 4. Общая функция `anon`

Вход: строка и список `[метка, start, end]`. Индексы начинаются с нуля, правая граница **не включается**, то есть сущность равна `text[start:end]`. Также поддерживается одна тройка, как в примере методички.

Основные метки: `NAME`, `SURNAME`, `EMAIL`, `INN`, `PHONE`, `CARD`. Допускаются русские названия и распространённые варианты меток, перечисленные в `LABELS`.

Сначала проверяются все интервалы, затем формируются замены. Вставка идёт справа налево, поэтому изменение длины имени не сдвигает координаты остальных сущностей. Текст вне интервалов остаётся прежним.

In [5]:
LABELS = {
    'NAME': 'NAME', 'FIRST_NAME': 'NAME', 'ИМЯ': 'NAME',
    'SURNAME': 'SURNAME', 'LAST_NAME': 'SURNAME', 'ФАМИЛИЯ': 'SURNAME',
    'EMAIL': 'EMAIL', 'E-MAIL': 'EMAIL',
    'INN': 'INN', 'ИНН': 'INN',
    'PHONE': 'PHONE', 'PHONE_NUMBER': 'PHONE', 'НОМЕР ТЕЛЕФОНА': 'PHONE',
    'CARD': 'CARD', 'BANK_CARD': 'CARD', 'CARD_NUMBER': 'CARD', 'НОМЕР БАНКОВСКОЙ КАРТЫ': 'CARD',
}


def anon(text, attributes):
    """Обезличить интервалы [метка, start, end]; end не включается."""
    if not isinstance(text, str):
        raise TypeError('Текст должен быть строкой')
    if not isinstance(attributes, (list, tuple)):
        raise TypeError('Атрибуты должны быть списком')
    if len(attributes) == 3 and isinstance(attributes[0], str):
        attributes = [attributes]  # Также принимается одна тройка из примера методички.
    spans = []
    for item in attributes:
        if not isinstance(item, (list, tuple)) or len(item) != 3:
            raise ValueError('Каждый атрибут должен иметь вид [метка, start, end]')
        label, start, end = item
        if not isinstance(label, str) or label.upper() not in LABELS:
            raise ValueError('Неизвестная метка атрибута')
        if type(start) is not int or type(end) is not int or not 0 <= start < end <= len(text):
            raise ValueError('Некорректные границы интервала')
        spans.append((start, end, LABELS[label.upper()]))
    spans.sort()
    if any(left[1] > right[0] for left, right in zip(spans, spans[1:])):
        raise ValueError('Интервалы атрибутов не должны пересекаться')

    replacements = []
    for start, end, label in spans:
        value = text[start:end]
        if label in ('NAME', 'SURNAME'):
            # Несколько однозначных предлогов помогают различать омонимичные формы.
            hints = {'к': 'datv', 'ко': 'datv', 'о': 'loct', 'об': 'loct',
                     'обо': 'loct', 'для': 'gent', 'без': 'gent', 'от': 'gent', 'у': 'gent'}
            match = re.search(r'([А-Яа-яЁё-]+)\s+$', text[:start])
            hint = None
            if match:
                previous = match[1]
                hint = hints.get(previous.lower())
                # Предлог может находиться перед соседним именем/фамилией.
                if hint is None and previous.istitle() and any(
                        ('Name' in p.tag or 'Surn' in p.tag) and p.is_known
                        for p in morph.parse(previous)):
                    before = re.search(r'([А-Яа-яЁё]+)\s+$', text[:match.start()])
                    hint = hints.get(before[1].lower()) if before else None
            new = anon_person(value, 'Name' if label == 'NAME' else 'Surn', hint)
        else:
            generator = {'EMAIL': anon_email, 'INN': anon_inn,
                         'PHONE': anon_phone, 'CARD': anon_card}[label]
            new = generator(value)
        replacements.append((start, end, new))

    # Индексы относятся к исходному тексту. Вставка справа налево их не смещает.
    for start, end, new in reversed(replacements):
        text = text[:start] + new + text[end:]
    return text

### 5. Пример со всеми шестью атрибутами

Координаты ниже составляются по известным фрагментам демонстрационного текста. Это пример входа из ЛР1, а не реализация распознавания. Используются учебные значения.

In [6]:
text = (
    'Иван Петров получил перевод. '
    'E-mail: ivan.petrov@example.com; ИНН: 770698959438; '
    'телефон: +7 (999) 123-45-67; карта: 4561 2612 1234 5467.'
)
known = [
    ('NAME', 'Иван'), ('SURNAME', 'Петров'),
    ('EMAIL', 'ivan.petrov@example.com'), ('INN', '770698959438'),
    ('PHONE', '+7 (999) 123-45-67'), ('CARD', '4561 2612 1234 5467'),
]
attributes = []
for label, value in known:
    start = text.index(value)
    attributes.append([label, start, start + len(value)])

print('Входные атрибуты:')
for label, start, end in attributes:
    print([label, start, end], '→', text[start:end])
print('\nИсходный текст:\n', text, sep='')
print('\nОбезличенный текст:\n', anon(text, attributes), sep='')
print('\nПовторный вызов:\n', anon(text, attributes), sep='')
print('\nОдна тройка из примера методички:')
print(anon('Иван получил кредит', ['NAME', 0, 4]))

Входные атрибуты:
['NAME', 0, 4] → Иван
['SURNAME', 5, 11] → Петров
['EMAIL', 37, 60] → ivan.petrov@example.com
['INN', 67, 79] → 770698959438
['PHONE', 90, 108] → +7 (999) 123-45-67
['CARD', 117, 136] → 4561 2612 1234 5467

Исходный текст:
Иван Петров получил перевод. E-mail: ivan.petrov@example.com; ИНН: 770698959438; телефон: +7 (999) 123-45-67; карта: 4561 2612 1234 5467.

Обезличенный текст:
Дмитрий Попов получил перевод. E-mail: qkxm.xhkutr@example.com; ИНН: 770657338784; телефон: +7 (999) 872-03-58; карта: 4561 2698 2734 5467.

Повторный вызов:
Андрей Орлов получил перевод. E-mail: crxp.utszzo@example.com; ИНН: 770636528495; телефон: +7 (999) 667-29-19; карта: 4561 2661 2497 5467.

Одна тройка из примера методички:
Сергей получил кредит


### 6. Проверка требований

Проверяются неизменяемые части и оформление числовых атрибутов, контрольные разряды, мужские и женские формы, все шесть падежей, множественное число, регистр, дефисы и неизменность текста за пределами интервалов. Ошибка любого условия прерывает ячейку через `assert`.

In [7]:
def span(text, label, value):
    start = text.index(value)
    return [label, start, start + len(value)]


def non_digits(value):
    return [(i, c) for i, c in enumerate(value) if c not in string.digits]


# Контрольные примеры не генерируются проверяемыми функциями.
assert valid_inn('770698959438')
assert valid_inn('7707083893')
assert not valid_inn('770698959439')
assert not valid_inn('0000000000')
assert valid_card('4561261212345467')
assert valid_card('4111111111111111')
assert not valid_card('4111111111111112')

# Сохранение формата, частей номера и контрольных разрядов: 200 замен каждого типа.
samples = {
    'EMAIL': ['ivan.petrov+7@Example.COM', 'a@пример.рф'],
    'INN': ['770698959438', '7707 08389-3'],
    'PHONE': ['+7 (999) 123-45-67', '8 495 123 45 67', '+1 (202) 555-0123'],
    'CARD': ['4561 2612 1234 5467', '4111-1111-1111-1111', '4222222222222'],
}
for label, values in samples.items():
    for i in range(200):
        old = values[i % len(values)]
        new = anon(old, [label, 0, len(old)])
        assert new != old
        if label == 'EMAIL':
            assert new.split('@')[1] == old.split('@')[1]
            assert len(new) == len(old)
        else:
            assert len(new) == len(old) and non_digits(new) == non_digits(old)
            a, b = digits(old), digits(new)
            if label == 'INN':
                assert b[:4] == a[:4] and valid_inn(b)
            elif label == 'PHONE':
                assert b[:4] == a[:4]
            else:
                assert b[:6] == a[:6] and b[-4:] == a[-4:] and valid_card(b)

# Длины карт 13–19 цифр; последняя цифра исходника вычисляется отдельно.
for length in range(13, 20):
    body = '456126' + '3' * (length - 7)
    old = next(body + d for d in string.digits if luhn_sum(body + d) % 10 == 0)
    for _ in range(20):
        new = anon_card(old)
        assert len(new) == length and new[:6] == old[:6] and new[-4:] == old[-4:]
        assert new != old and valid_card(new)

# Грамматические свойства задаются явно, а не берутся из результата генератора.
morph_cases = [
    ('Иван', 'NAME', {'masc', 'sing', 'nomn'}),
    ('Ивана', 'NAME', {'masc', 'sing', 'gent'}),
    ('Ивану', 'NAME', {'masc', 'sing', 'datv'}),
    ('Ивана', 'NAME', {'masc', 'sing', 'accs'}),
    ('Иваном', 'NAME', {'masc', 'sing', 'ablt'}),
    ('Иване', 'NAME', {'masc', 'sing', 'loct'}),
    ('Анна', 'NAME', {'femn', 'sing', 'nomn'}),
    ('Анны', 'NAME', {'femn', 'sing', 'gent'}),
    ('Анне', 'NAME', {'femn', 'sing', 'datv'}),
    ('Анну', 'NAME', {'femn', 'sing', 'accs'}),
    ('Анной', 'NAME', {'femn', 'sing', 'ablt'}),
    ('Анне', 'NAME', {'femn', 'sing', 'loct'}),
    ('Саша', 'NAME', {'ms-f', 'sing', 'nomn'}),
    ('Иваны', 'NAME', {'masc', 'plur', 'nomn'}),
    ('Иванов', 'SURNAME', {'masc', 'sing', 'nomn'}),
    ('Иванову', 'SURNAME', {'masc', 'sing', 'datv'}),
    ('Ивановым', 'SURNAME', {'masc', 'sing', 'ablt'}),
    ('Иванова', 'SURNAME', {'femn', 'sing', 'nomn'}),
    ('Ивановой', 'SURNAME', {'femn', 'sing', 'gent'}),
    ('Иванову', 'SURNAME', {'femn', 'sing', 'accs'}),
    ('Ивановы', 'SURNAME', {'plur', 'nomn'}),
    ('Ивановых', 'SURNAME', {'plur', 'gent'}),
    ('Ивановым', 'SURNAME', {'plur', 'datv'}),
    ('Ивановыми', 'SURNAME', {'plur', 'ablt'}),
]
for old, label, required in morph_cases:
    kind = 'Name' if label == 'NAME' else 'Surn'
    # Падеж помогает разобрать неоднозначную форму; в anon подсказка берётся из предлога.
    case = next(x for x in required if x in {'nomn', 'gent', 'datv', 'accs', 'ablt', 'loct'})
    for _ in range(5):
        new = anon_person(old, kind, case)
        assert new != old
        assert any(kind in p.tag and required.issubset(p.tag.grammemes) for p in morph.parse(new)), (old, new, required)

for old in ['иван', 'ИВАН', 'Анна-Мария', 'Сидоров-Петров']:
    label = 'SURNAME' if old == 'Сидоров-Петров' else 'NAME'
    new = anon(old, [label, 0, len(old)])
    assert old != new and old.count('-') == new.count('-')
    assert (old.islower(), old.isupper(), old.istitle()) == (new.islower(), new.isupper(), new.istitle())

for old in ['к Анне', 'об Анне', 'для Анны', 'о Петровой', 'к Ивану Петрову']:
    words = old.split()
    label = 'SURNAME' if words[-1].startswith('Петров') else 'NAME'
    item = span(old, label, words[-1])
    new = anon(old, [item])
    assert new.startswith(old[:item[1]]) and new != old
    case = {'к': 'datv', 'об': 'loct', 'для': 'gent', 'о': 'loct'}[words[0]]
    assert any(p.tag.case == case for p in morph.parse(new.split()[-1]))

# Проверка, что вне интервалов, в том числе между соседними, текст не меняется.
old = 'До: Иван/Петров; email=a@example.com. После!'
items = [span(old, 'NAME', 'Иван'), span(old, 'SURNAME', 'Петров'),
         span(old, 'EMAIL', 'a@example.com')]
new = anon(old, list(reversed(items)))
assert re.fullmatch(r'До: [А-Яа-яЁё]+/[А-Яа-яЁё]+; email=[a-z]@example\.com\. После!', new)
assert anon('', []) == ''
assert anon('Текст без ПДн', []) == 'Текст без ПДн'
assert anon('ИванПетров', [['NAME', 0, 4], ['SURNAME', 4, 10]]) != 'ИванПетров'
print('Проверки структуры, морфологии и контрольных разрядов пройдены.')

Проверки структуры, морфологии и контрольных разрядов пройдены.


### 7. Реакция на неожиданные значения

Неверный тип текста или списка вызывает `TypeError`. Неправильная разметка и неподдерживаемое значение вызывают `ValueError`. Пересечения и дубликаты интервалов отклоняются. Пустой список атрибутов возвращает исходный текст.

Поддерживаются обычные e-mail без кавычек в логине, телефоны из 10–15 цифр, ИНН из 10/12 цифр и карты из 13–19 цифр. Имена и фамилии должны быть русскими словами из морфологического словаря, в том числе с дефисами. Инициалы, неизвестные слова и смешанный регистр отклоняются. Маскированные и невалидные исходные номера также отклоняются. Корректность номера означает проверку формата и контрольных разрядов; существование номера в реестре не проверяется.

Если обнаружена ошибка, `anon` не возвращает частично обработанный текст. В примерах ниже ошибки намеренно перехватываются, чтобы показать сообщения и продолжить выполнение ноутбука.

In [8]:
def expect_error(title, action):
    try:
        action()
    except (ValueError, TypeError) as error:
        print(f'{title}: {error}')
    else:
        raise AssertionError(f'Ожидалась ошибка: {title}')


bad_inputs = [
    ('Текст не строка', lambda: anon(None, [])),
    ('Вместо списка передан None', lambda: anon('Иван', None)),
    ('Неполная тройка', lambda: anon('Иван', [['NAME', 0]])),
    ('Неизвестная метка', lambda: anon('Иван', ['UNKNOWN', 0, 4])),
    ('Отрицательный индекс', lambda: anon('Иван', ['NAME', -1, 4])),
    ('Выход за границы', lambda: anon('Иван', ['NAME', 0, 20])),
    ('Пустой интервал', lambda: anon('Иван', ['NAME', 0, 0])),
    ('Индекс не целый', lambda: anon('Иван', ['NAME', 0.0, 4])),
    ('Пересекающиеся интервалы', lambda: anon('Иван', [['NAME', 0, 4], ['NAME', 1, 4]])),
    ('Дублирующиеся интервалы', lambda: anon('Иван', [['NAME', 0, 4], ['NAME', 0, 4]])),
    ('Слово не является именем', lambda: anon('стол', ['NAME', 0, 4])),
    ('Инициал', lambda: anon('И.', ['NAME', 0, 2])),
    ('Смешанный регистр', lambda: anon('иВаН', ['NAME', 0, 4])),
    ('Почта без @', lambda: anon_email('user.example.com')),
    ('Почта с двумя @', lambda: anon_email('user@@example.com')),
    ('Пустой логин', lambda: anon_email('@example.com')),
    ('Неправильный домен', lambda: anon_email('user@-example.com')),
    ('Короткий ИНН', lambda: anon_inn('123')),
    ('Неверный контрольный разряд ИНН', lambda: anon_inn('770698959439')),
    ('Буквы вместо ИНН', lambda: anon_inn('7706ABC59438')),
    ('Короткий телефон', lambda: anon_phone('+7 (999)')),
    ('Буквы в телефоне', lambda: anon_phone('телефон неизвестен')),
    ('Частичная карта', lambda: anon_card('**** **** **** 1111')),
    ('Неверная проверка Луна', lambda: anon_card('4111 1111 1111 1112')),
]
for title, action in bad_inputs:
    expect_error(title, action)
print(f'Все {len(bad_inputs)} проверок неожиданных значений пройдены.')

Текст не строка: Текст должен быть строкой
Вместо списка передан None: Атрибуты должны быть списком
Неполная тройка: Каждый атрибут должен иметь вид [метка, start, end]
Неизвестная метка: Неизвестная метка атрибута
Отрицательный индекс: Некорректные границы интервала
Выход за границы: Некорректные границы интервала
Пустой интервал: Некорректные границы интервала
Индекс не целый: Некорректные границы интервала
Пересекающиеся интервалы: Интервалы атрибутов не должны пересекаться
Дублирующиеся интервалы: Интервалы атрибутов не должны пересекаться
Слово не является именем: Имя/фамилия не найдены в морфологическом словаре
Инициал: Имя/фамилия: ожидается русское слово или части через дефис
Смешанный регистр: Имя/фамилия: смешанный регистр не поддерживается
Почта без @: EMAIL: требуется один символ @
Почта с двумя @: EMAIL: требуется один символ @
Пустой логин: EMAIL: ожидается обычный адрес вида user@example.com
Неправильный домен: EMAIL: ожидается обычный адрес вида user@example.com
Коротки

### 8. Проверка случайности

Для каждого атрибута выполняются 20 независимых вызовов. Считается число разных замен; оно должно быть больше одного. Дополнительно проверяется, что ни одна замена не совпала с исходником. При новом выполнении ячейки значения генерируются заново.

In [9]:
for label, old in known:
    results = [anon(old, [label, 0, len(old)]) for _ in range(20)]
    assert old not in results
    assert len(set(results)) > 1, f'{label}: замены не меняются'
    print(f'{label}: {len(set(results))} разных значений из {len(results)} вызовов')
    print('  Примеры:', ', '.join(results[:4]))
print('Проверка случайности пройдена.')

NAME: 7 разных значений из 20 вызовов
  Примеры: Сергей, Николай, Пётр, Сергей
SURNAME: 10 разных значений из 20 вызовов
  Примеры: Морозов, Новиков, Орлов, Сидоров
EMAIL: 20 разных значений из 20 вызовов
  Примеры: tlil.eqabzh@example.com, ebvq.btebxc@example.com, buhv.npxjmo@example.com, ntlv.xzcnlm@example.com
INN: 20 разных значений из 20 вызовов
  Примеры: 770670658984, 770622768881, 770694280003, 770679671669
PHONE: 20 разных значений из 20 вызовов
  Примеры: +7 (999) 382-93-03, +7 (999) 630-47-49, +7 (999) 642-73-64, +7 (999) 698-15-89
CARD: 20 разных значений из 20 вызовов
  Примеры: 4561 2688 7693 5467, 4561 2658 3615 5467, 4561 2600 6765 5467, 4561 2645 7745 5467
Проверка случайности пройдена.


### 9. Свой текст и результат ЛР1

Замените `my_text` и `my_attributes` своими значениями. Координаты должны относиться к исходному тексту. После изменения текста координаты также нужно обновить. Функция возвращает строку.

In [10]:
my_text = 'Анна переводит Ивану 1000 рублей.'
my_attributes = [['NAME', 0, 4], ['NAME', 15, 20]]

print('Было:', my_text)
print('Стало:', anon(my_text, my_attributes))

# При наличии функции распознавания из ЛР1:
# my_attributes = recognize(my_text)
# result = anon(my_text, my_attributes)

Было: Анна переводит Ивану 1000 рублей.
Стало: Елена переводит Петру 1000 рублей.


### Вывод

Реализована функция `anon` для синонимического обезличивания шести требуемых атрибутов. Замены сохраняют предусмотренные методичкой части значений, морфологические признаки выбранного разбора и контрольные разряды. Проверены корректные входные данные, случайность замен и обработка ошибочных значений.

При защите можно показать пример из раздела 5, проверки из разделов 6–8 и объяснить: `pymorphy3` отвечает за разбор и склонение; `SystemRandom` — за случайный выбор; ИНН проверяется взвешенными суммами, карта — алгоритмом Луна; замены вставляются справа налево.

### Материалы

- «Лабораторная работа 2.docx» — требования и рекомендации к выполнению.
- «Лабораторная работа 1.docx» — формат входных интервалов.
- Лекция 3 — морфология и обезличивание структурированных значений, в частности сохранение контрольной цифры карты.
- [Документация pymorphy3: разбор и склонение](https://pymorphy3.readthedocs.io/en/latest/user/guide.html).